# Lab 12 — Controlled Evasion-Style Robustness Testing

*AI Cybersecurity Masterclass · Lab 12 · AI60secLab*

**Companion video:** [▶ How Hackers Fool AI Models](https://www.youtube.com/watch?v=deKvWTAQ0To&list=PLG4l6Ssi9A00) · [▶ Can a Small Change Make Cybersecurity AI Miss an Attack?](https://www.youtube.com/watch?v=FPYX_cPciUw&list=PLG4l6Ssi9A00)

**Research question:** How stable is attack recall when selected numeric telemetry values change slightly?

**Scope:** offline defensive research on copied public data.

This notebook does not search for an optimal bypass. It uses bounded, non-targeted perturbations to measure sensitivity.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import recall_score

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.adversarial import (
    available_numeric_attack_features,
    training_bounds,
    bounded_random_perturbation,
    augment_attacks_with_noise,
)

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.fit(X_train, y_train)

perturbation_columns = available_numeric_attack_features(X_train)
bounds = training_bounds(
    X_train,
    perturbation_columns,
)

print("Perturbation columns:")
print(perturbation_columns)

In [ ]:
y_test_reset = y_test.reset_index(drop=True)
X_test_reset = X_test.reset_index(drop=True)

attack_mask = y_test_reset == 1
X_attack = (
    X_test_reset.loc[attack_mask]
    .reset_index(drop=True)
)

clean_recall = recall_score(
    [1] * len(X_attack),
    model.predict(X_attack),
    zero_division=0,
)

print("Clean attack recall:", round(clean_recall, 4))

In [ ]:
rows = []

for epsilon in [0.00, 0.01, 0.05, 0.10, 0.20]:
    stressed = (
        X_attack.copy()
        if epsilon == 0
        else bounded_random_perturbation(
            X_attack,
            columns=perturbation_columns,
            epsilon=epsilon,
            bounds=bounds,
            random_state=42,
        )
    )

    prediction = model.predict(stressed)
    score = model.predict_proba(stressed)[:, 1]

    recall = recall_score(
        [1] * len(stressed),
        prediction,
        zero_division=0,
    )

    rows.append({
        "epsilon": epsilon,
        "attack_recall": recall,
        "mean_attack_score": float(score.mean()),
        "miss_rate": 1 - recall,
    })

robustness = pd.DataFrame(rows)
display(robustness.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    robustness["epsilon"],
    robustness["attack_recall"],
    marker="o",
)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Bounded perturbation epsilon")
ax.set_ylabel("Attack recall")
ax.set_title("Robustness under non-targeted input perturbation")
plt.show()

## Step 2: The true-positive protocol

The question an evasion test asks is: *can a small change make the model miss an attack it already caught?*

So start only from attacks the model **detected** on clean data (true positives), copy them, perturb the copies, and count how many slip through. The model stays fixed. Only the input changes. Labels never change.

Repeat with several random seeds and report the spread, so one lucky draw cannot carry the conclusion.

In [ ]:
import numpy as np

clean_prediction = model.predict(X_attack)
X_true_positive = X_attack.loc[clean_prediction == 1].reset_index(drop=True)
print("Attacks detected on clean data (true positives):", len(X_true_positive))

seeds = [0, 1, 2, 3, 4]
tp_rows = []

for epsilon in [0.01, 0.05, 0.10, 0.20]:
    for seed in seeds:
        stressed = bounded_random_perturbation(
            X_true_positive,
            columns=perturbation_columns,
            epsilon=epsilon,
            bounds=bounds,
            random_state=seed,
        )
        still_detected = model.predict(stressed)
        tp_rows.append({
            "epsilon": epsilon,
            "seed": seed,
            "evasion_rate": float((still_detected == 0).mean()),
        })

tp_results = pd.DataFrame(tp_rows)
tp_summary = (
    tp_results.groupby("epsilon")["evasion_rate"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
display(tp_summary.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    tp_summary["epsilon"],
    tp_summary["mean"],
    yerr=tp_summary["std"],
    marker="o",
    capsize=4,
)
ax.set_xlabel("Bounded perturbation epsilon")
ax.set_ylabel("Share of former true positives now missed")
ax.set_title("Evasion rate on previously detected attacks (mean ± std, 5 seeds)")
plt.show()

**Reading this result honestly:** random bounded noise measures *sensitivity*. A real attacker controls packets and timing, not feature values, and many features cannot move independently without breaking the traffic. A low evasion rate here does not mean the model resists a real bypass. It means it is not fragile to small, undirected changes.

## Defensive augmentation experiment

Add small noisy copies of attack rows to the **training data**, retrain, and compare robustness. This is an experiment, not a guaranteed defense.

In [ ]:
X_augmented, y_augmented = augment_attacks_with_noise(
    X_train,
    y_train,
    columns=perturbation_columns,
    epsilon=0.05,
    copies=1,
    positive_label=1,
    bounds=bounds,
    random_state=42,
)

robust_model = random_forest_pipeline(
    X_augmented
)
robust_model.fit(
    X_augmented,
    y_augmented,
)

comparison_rows = []

for epsilon in [0.00, 0.05, 0.10, 0.20]:
    stressed = (
        X_attack.copy()
        if epsilon == 0
        else bounded_random_perturbation(
            X_attack,
            columns=perturbation_columns,
            epsilon=epsilon,
            bounds=bounds,
            random_state=99,
        )
    )

    base_recall = recall_score(
        [1] * len(stressed),
        model.predict(stressed),
        zero_division=0,
    )

    robust_recall = recall_score(
        [1] * len(stressed),
        robust_model.predict(stressed),
        zero_division=0,
    )

    comparison_rows.append({
        "epsilon": epsilon,
        "baseline_recall": base_recall,
        "augmented_training_recall": robust_recall,
    })

defense_comparison = pd.DataFrame(
    comparison_rows
)
display(defense_comparison.round(4))

## Research interpretation

- Does confidence decline before binary recall declines?
- Which perturbation level produces a meaningful change?
- Did noisy training augmentation help?
- Did it preserve clean performance?
- Why is this non-targeted stress test different from an optimized evasion attack?

**Takeaway:** clean benchmark performance and robustness are different properties.